In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
import numpy as np

from cmmdft.units_constants import *

from cmmdft.system import NanoporousHost, SphericalLJGuest, DualModelGuest, EmptyHost, GuestMixture
from cmmdft.program import Program
from cmmdft.solver import Picard, Anderson, QuasiNewton
from cmmdft.calculator import Calculator
from cmmdft.eos import PCSAFTEOS, PengRobinsonMixEOS
from cmmdft.guest_dict import SAFT_pars_dict, chk_dict, pars_dict, LJ_pars_dict

ModuleNotFoundError: No module named 'cmmdft'

In [19]:
crit_parameters_dict = {'CH4' :  (16.031*amu, 190.56*kelvin, 45.99*atm, 0.01142),
                        'C2H4' : (30.047*amu, 282.34*kelvin, 50.41*atm, 0.0866),
                        'C2H6' : (28.031*amu, 305.32*kelvin, 48.718*atm, 0.0993),
                        'C3H6' : (42.047*amu, 364.9*kelvin, 45.79*atm, 0.144),
                        'C3H8' : (44.063*amu, 369.8*kelvin, 42.6*atm, 0.1530),
                        'CO2' :  (43.990*amu, 304.25*kelvin, 73.8*atm, 0.225)}

## System setup

In [ ]:
guestname = 'C2H6_UA'
mass, sigma, epsilon, m = (30.047*amu, 3.51681*angstrom, 191.45389*boltzmann, 1.60686)

guestchk = 'InputFiles/Guest/C2H6/struct_CA.chk'
guestpar = 'InputFiles/Guest/C2H6/pars_vdw_CA_trappe.txt'
guest = DualModelGuest(guestname, mass, sigma, epsilon, struct=guestchk, par=guestpar, m=m, ffname='TRAPPE-PCSAFT', hs_def='exp')

hostname = 'ZIF8'
hostchk = 'InputFiles/Host/ZIF8/ZIF8_opened_struct.pdb'
hostpar = 'InputFiles/Host/ZIF8/AMBER_system.xml'
host = NanoporousHost(hostname, hostchk, hostpar,
                      unit_energy='kjmol', unit_distance='angstrom', unit_sigma='nanometer', unit_mass='amu') #it is important to specify the units of the input file

ff_suffix = 'AMBER_TRAPPE'
funct_suffix = 'atWBII_PCSAFT'

grid_suffix = '0_25' # a greater resolution is required for calculation of free energy profiles


 SYSTEM  Reading guest from InputFiles/Guest/C2H6/struct_CA.chk with parameters from InputFiles/
 SYSTEM  Guest/C2H6/pars_vdw_CA_trappe.txt

 SYSTEM  Reading host structure from InputFiles/Host/ZIF8/ZIF8_struct.chk with parameters from 
 SYSTEM  InputFiles/Host/ZIF8/pars_vdw_AMBER.txt


In [22]:
# thermodynamic properties
temperature = 300*kelvin
pressures = np.linspace(0.1,50,10)*bar

In [ ]:
# program initialization
program = Program(prefix=f'OutputFiles/DiffusionPathExample', hostname=hostname, guestname=guestname, ff_suffix=ff_suffix, funct_suffix=funct_suffix, grid_suffix=grid_suffix)

program.set_system(host, guest)
program.set_grid(spacing = 0.5*angstrom)
program.set_eos('PCSAFT')
program.init_free_energy(temperature)

program.fener.add_external_potential(temperature)
program.fener.add_hard_sphere()
program.fener.add_PCSAFT()
program.set_temperature(temperature)

#defining multiple convergence criteria
criteria = ['riue', 'res']
thresholds = [1e-7, 1e-6]

solver = Anderson(program, criterion=criteria, threshold=thresholds)
program.set_solver()

calc = Calculator(program)


 PROGRAM Initializing work directory OutputFiles2/ZIF8/C2H6_UA/AMBER_TRAPPE/atWBII_PCSAFT/0_50

 GRID    Initializing grid
 GRID      number of grid points  =    16,    16,    16
 GRID      spacing of grid points = 1.062, 1.062, 1.062 A

 PROGRAM eos set to PCSAFT

 GUEST   Computing Rhs and Rhs_zero at 300.00000 without storing...
 GUEST     Rhs =   1.73 A  -  Vhs =  21.59 A**3

 FREEENE Initializing atWBII functional for hard-sphere contribution

 FREEENE Initializing PC-SAFT functional for attractive and repulsive interaction contribution

 GUEST   Computing Rhs and Rhs_zero at 300.00000 without storing...
 GUEST     Rhs =   1.73 A  -  Vhs =  21.59 A**3

 FREEENE Setting temperature for functional HardSphere
 FREEENE Setting temperature for functional PCSAFT


In [ ]:
for P in pressures:
    program.solve(pressure=P)
    calc.save_loading_and_grand_potential(temperature=temperature, pressure=P) # Saves loading and grand potential per calculation to an npz file for later use (necessary for later integration)




 GUEST   Computing Rhs and Rhs_zero at 300.00000 without storing...
 GUEST     Rhs =   1.73 A  -  Vhs =  21.59 A**3

 FREEENE Setting temperature for functional HardSphere
 FREEENE Setting temperature for functional PCSAFT

 PROGRAM Thermodynamic conditions:
 PROGRAM   temperature = 300.000   K
 PROGRAM   chem. pot.  = -44.936 kJ/mol
 PROGRAM   fugacity    =   0.100 bar
 PROGRAM Setting initial guess for density at 3.580e-07/cellvolume in pores

 ANDERSO step   1/100 *  Loading                           =  1.1852e-02 mol./uc
 ANDERSO              *  Abs. Integr. Unsign. Err. density =  2.2807e-11 mol./uc
 ANDERSO              *  Rel. Integr. Unsign. Err. density =  1.9244e-09 
 ANDERSO              *  Grand potential                   = -2.9540e-02 kJ/mol 

 SOLVER  #################################################################################
 SOLVER  Calculated the density for a chemical potential of -44.936 kJ/mol in 0.52 seconds
 SOLVER  ########################################

## Defining of the diffusion axis

In [ ]:
carbon_indices = np.array([171,191,175,187,179,183]) # indices of the atoms defining the gate opening
step_dist = program.grid.spacings[0]/np.sqrt(3) # because the diffusion axis is the 111 diagonal for this unit cell (adjust as needed)
distance_from_axis = 7*angstrom # necessary to keep the integration to the relevant pores (adjust as needed)
cvs_limits = np.array([-7.5,7.5])*angstrom # defines the edges of the cv range

cvs, cvs_mat, dist_mask = calc.collective_variable(ring_indices=carbon_indices, dist_from_axis=distance_from_axis,
                                                    step_dist=step_dist, cvs_limits=cvs_limits, supercell=True)

In [ ]:
for P in pressures:
    calc.project_density(cvs, cvs_mat, dist_mask, temperature=temperature, pressure=P)
    calc.free_energy_path(temperature=temperature, pressure=P)

# Guest mixture

In [ ]:
guestname1 = 'C2H6_UA'
mass1, sigma1, epsilon1, m1 = (30.047*amu, 3.51681*angstrom, 191.45389*boltzmann, 1.60686)

guestchk1 = 'InputFiles/Guest/C2H6/struct_CA.chk'
guestpar1 = 'InputFiles/Guest/C2H6/pars_vdw_CA_trappe.txt'
guest1 = DualModelGuest(guestname1, mass1, sigma1, epsilon1, struct=guestchk1, par=guestpar1, m=m1, ffname='TRAPPE-PCSAFT', hs_def='exp')

guestname2 = 'C2H4_UA'
mass2, sigma2, epsilon2, m2 = (28.031*amu, 3.40769*angstrom, 177.32571*boltzmann, 1.59196)

guestchk2 = 'InputFiles/Guest/C2H6/struct_CA.chk'
guestpar2 = 'InputFiles/Guest/C2H6/pars_vdw_CA_trappe.txt'
guest2 = DualModelGuest(guestname2, mass2, sigma2, epsilon2, struct=guestchk2, par=guestpar2, m=m2, ffname='TRAPPE-PCSAFT', hs_def='exp')

mole_fractions = [0.5, 0.5]
mixture = GuestMixture([guest1, guest2], mole_fractions)

hostname = 'ZIF8'
hostchk = 'InputFiles/Host/ZIF8/ZIF8_struct.chk'
hostpar = 'InputFiles/Host/ZIF8/pars_vdw_AMBER.txt'
host = NanoporousHost(hostname, hostchk, hostpar)

ff_suffix = 'AMBER_TRAPPE'
funct_suffix = 'tWBII_PCSAFT'

grid_suffix = '0_50'

In [ ]:
# program initialization
program = Program(prefix=f'OutputFiles', hostname=hostname, guestname=mixture.name, ff_suffix=ff_suffix, funct_suffix=funct_suffix, grid_suffix=grid_suffix)

program.set_system(host, guest)
program.set_grid(spacing = 0.5*angstrom)
program.set_eos('PCSAFT')
program.init_free_energy(temperature)

program.fener.add_external_potential(temperature)
program.fener.add_hard_sphere()
program.fener.add_PCSAFT()
program.set_temperature(temperature)

#defining multiple convergence criteria
criteria = ['riue', 'res']
thresholds = [1e-7, 1e-6]

solver = Anderson(program, criterion=criteria, threshold=thresholds)
program.set_solver()

In [ ]:
program.adsorption_isotherm(temperature, pressures, rewrite=True)